# Day 2: Multi-File Data Processing Pipeline

**Scenario.** Soko Fresh Supermarkets runs branches in Nairobi, Mombasa, Kisumu and Nakuru. Every
night each branch's till system drops a CSV of the day's sales into a shared folder. The files are not
as clean as finance assumes: some fields are blank, some rows are re-sent from earlier days, and a
recent POS upgrade renamed one column.

This notebook discovers every daily file in `data/`, validates and cleans each row, combines the good
rows into one trusted CSV, keeps every rejected row with the reasons it was refused, and writes a
processing summary:

- `output/consolidated_transactions.csv`
- `output/rejected_transactions.csv`
- `output/summary.txt`

Each of the eight tasks has its own section. The functions keep the signatures from `starter.py`.

## Setup

In [1]:
# importing libraries
import csv                       # read and write CSV files row by row (no pandas while cleaning)
import logging                   # warnings about problem files
import shutil                    # copy files for the "fourth day" test
import tempfile                  # throwaway folders for the tests
from datetime import datetime    # timestamp for the summary
from pathlib import Path         # find files and build paths that work on any OS

import pandas as pd              # only used at the end, to check the outputs reconcile

# settings
INPUT_DIR = Path("data")
OUTPUT_DIR = Path("output")
FILE_PATTERN = "transactions_*.csv"

# columns every file must have
REQUIRED_COLUMNS: set[str] = {
    "transaction_id", "date", "branch", "customer_id",
    "product", "quantity", "unit_price", "payment_method",
}
# columns that must hold positive whole numbers
NUMERIC_COLUMNS: set[str] = {"quantity", "unit_price"}

# known renames from the till system: alias in the file -> the name we use
COLUMN_ALIASES: dict[str, str] = {"qty": "quantity"}

# output layout, taken from expected_output.md
CONSOLIDATED_COLUMNS = [
    "transaction_id", "date", "branch", "customer_id", "product",
    "quantity", "unit_price", "line_total", "payment_method", "payment_ref", "source_file",
]
REJECTED_LEAD_COLUMNS = ["source_file", "row_number", "transaction_id", "reasons"]

# required columns in a fixed order, so reasons always come out in the same order
REQUIRED_ORDER = [col for col in CONSOLIDATED_COLUMNS if col in REQUIRED_COLUMNS]

# every reason starts with one of these prefixes; the summary counts rows per category
CATEGORY_LABELS = {
    "missing value": "missing values",
    "invalid number": "invalid numbers",
    "duplicate ID": "duplicate IDs",
    "malformed row": "malformed rows",
    "file column issue": "file column issues",
}

# show warnings as one plain line each
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("pipeline")

## Task 1: Read multiple CSV files

`read_transactions()` reads one daily file into its header and a list of row dicts. Every row is
tagged with its `source_file` and `row_number` (the line in the file, where the header is line 1), so
the rejected-records report can point finance at the exact line to chase.

- Files are opened with `utf-8-sig` so a byte-order mark from Excel does not end up glued to the first
  column name.
- If a row has more values than the header has columns, the extras are kept in `extra_values` so
  the row can be rejected as malformed instead of losing data.

In [2]:
def read_transactions(path: Path) -> tuple[list[str], list[dict]]:
    """Read one file and return (header, rows). Every row remembers its source file and line."""
    path = Path(path)
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        # strip stray spaces from column names (" quantity" -> "quantity")
        header = [col.strip() for col in (reader.fieldnames or [])]
        reader.fieldnames = header

        rows = []
        for row in reader:
            row = dict(row)
            # DictReader puts surplus values under the key None
            extra = row.pop(None, None)
            if extra:
                row["extra_values"] = "|".join(extra)
            row["source_file"] = path.name
            row["row_number"] = reader.line_num   # line in the file, header is line 1
            rows.append(row)
    return header, rows


# reading the first day's file
header, rows = read_transactions(INPUT_DIR / "transactions_2026-10-01.csv")
print("Header:", header)
print(f"Rows read: {len(rows)}")
rows[0]

Header: ['transaction_id', 'date', 'branch', 'customer_id', 'product', 'quantity', 'unit_price', 'payment_method', 'payment_ref']
Rows read: 15


{'transaction_id': 'TXN-1001',
 'date': '2026-10-01',
 'branch': 'Nairobi CBD',
 'customer_id': 'CUST-201',
 'product': 'Unga Pembe 2kg',
 'quantity': '3',
 'unit_price': '210',
 'payment_method': 'M-Pesa',
 'payment_ref': 'QJA1X2K9PL',
 'source_file': 'transactions_2026-10-01.csv',
 'row_number': 2}

## Task 2: Discover files with pathlib

`discover_files()` uses `Path.glob()` to pick up every file matching `transactions_*.csv`. The dates in
the file names are in ISO format (`YYYY-MM-DD`), so sorting the names as text also sorts them by date.
A new day's file is picked up automatically, with no code change.

In [3]:
def discover_files(folder: Path, pattern: str = FILE_PATTERN) -> list[Path]:
    """Return the matching CSV files in a stable (date) order using pathlib."""
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f"Input folder not found: {folder.resolve()}")
    # ISO dates in the names (2026-10-01) sort correctly as plain text
    return sorted((p for p in folder.glob(pattern) if p.is_file()), key=lambda p: p.name)


# discovering the daily files
files = discover_files(INPUT_DIR)
for path in files:
    print(path)

data\transactions_2026-10-01.csv
data\transactions_2026-10-02.csv
data\transactions_2026-10-03.csv


In [4]:
# testing: drop a fourth day's file (and an unrelated file) into a copy of the folder
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    for path in files:
        shutil.copy(path, tmp)
    shutil.copy(files[0], tmp / "transactions_2026-10-04.csv")
    (tmp / "notes.txt").write_text("not a transactions file")
    print([p.name for p in discover_files(tmp)])

['transactions_2026-10-01.csv', 'transactions_2026-10-02.csv', 'transactions_2026-10-03.csv', 'transactions_2026-10-04.csv']


## Task 3: Validate required columns

`validate_columns()` is a set difference: required columns minus the columns the file has.

**Decision for a file with the wrong header: map a known alias, reject anything else.**

- `transactions_2026-10-03.csv` has `qty` instead of `quantity`, plus a new `cashier` column. That
  comes from the POS upgrade: the data is the same, only the column name changed. Rejecting the file
  would throw away a whole day of valid sales and leave finance chasing every branch for nothing.
- Only renames listed in `COLUMN_ALIASES` are mapped, and only when the real column is missing, and
  every mapping is reported in the summary. Guessing at unknown names could put the wrong data in
  the wrong column.
- If a required column is still missing after mapping, **the whole file is rejected**: every row goes to
  the rejected file with the reason, so nothing disappears.
- Extra columns such as `cashier` are not part of the agreed output, so they are left out of the
  consolidated file (they are kept in the rejected file).

In [5]:
def validate_columns(header: list[str], required: set[str] = REQUIRED_COLUMNS) -> set[str]:
    """Return the set of required columns missing from the header (empty set means valid)."""
    return set(required) - set(header)


def apply_aliases(header: list[str], rows: list[dict]) -> tuple[list[str], dict[str, str]]:
    """Rename known aliases (e.g. qty -> quantity) when the real column is missing.

    Returns the new header and the renames that were applied.
    """
    missing = validate_columns(header)
    # only map an alias that is present AND stands in for a column that is missing
    renames = {old: new for old, new in COLUMN_ALIASES.items() if old in header and new in missing}
    if renames:
        header = [renames.get(col, col) for col in header]
        for row in rows:
            for old, new in renames.items():
                row[new] = row.pop(old)
    return header, renames


# checking every file's header
for path in files:
    header, rows = read_transactions(path)
    missing = validate_columns(header)
    extra = set(header) - REQUIRED_COLUMNS
    print(f"{path.name}: missing {missing or 'nothing'}, extra {extra}")

transactions_2026-10-01.csv: missing nothing, extra {'payment_ref'}
transactions_2026-10-02.csv: missing nothing, extra {'payment_ref'}
transactions_2026-10-03.csv: missing {'quantity'}, extra {'cashier', 'qty', 'payment_ref'}


In [6]:
# mapping the alias in the 3 October file
header, rows = read_transactions(INPUT_DIR / "transactions_2026-10-03.csv")
header, renames = apply_aliases(header, rows)
print("Renamed:", renames)
print("Missing after mapping:", validate_columns(header) or "nothing")

Renamed: {'qty': 'quantity'}
Missing after mapping: nothing


## Task 4: Find missing values and duplicate transaction IDs

`find_problems()` returns **every** reason a row is invalid, so one row can carry several reasons:

- **missing value**: a required field is blank (`payment_ref` is optional, since cash sales have none)
- **invalid number**: `quantity` or `unit_price` is not a whole number, or is zero or negative
- **duplicate ID**: the `transaction_id` was already accepted earlier in this run, in this file or an
  earlier one
- **malformed row**: the row has more values than the header has columns

**Which copy of a duplicate is kept:** the **first** one accepted. Files are processed in date order,
so that is the original day's record; later copies are re-sends (their `date` still shows the original
day). `seen_ids` only holds IDs that were *accepted*, so if the first copy was rejected (say for a
blank field), a later valid copy is not thrown away as a duplicate.

In [7]:
def find_problems(row: dict, seen_ids: set[str]) -> list[str]:
    """Return a list of human-readable reasons this row is invalid (empty list means valid).

    Covers blank required values, non-numeric or non-positive quantity/price,
    and transaction IDs already accepted in this run.
    """
    problems = []

    # blank required fields (None means the row was too short)
    for col in REQUIRED_ORDER:
        if not (row.get(col) or "").strip():
            problems.append(f"missing value: {col}")

    # quantity and price must be positive whole numbers
    for col in sorted(NUMERIC_COLUMNS):
        value = (row.get(col) or "").strip()
        if not value:
            continue   # already reported as missing
        try:
            number = int(value)
        except ValueError:
            problems.append(f"invalid number: {col} '{value}' is not a whole number")
            continue
        if number <= 0:
            problems.append(f"invalid number: {col} {number} is not positive")

    # duplicate ID: already accepted earlier in this run
    txn_id = (row.get("transaction_id") or "").strip()
    if txn_id and txn_id in seen_ids:
        problems.append(f"duplicate ID: {txn_id} was already accepted earlier in this run")

    # surplus values that did not fit under any column
    if row.get("extra_values"):
        problems.append("malformed row: more values than columns in the header")

    return problems


# testing: a row with several problems at once, and a clean row
messy = {"transaction_id": "TXN-1001", "date": "2026-10-04", "branch": " ",
         "customer_id": "CUST-201", "product": "Kimbo 1kg", "quantity": "two",
         "unit_price": "-520", "payment_method": "Cash", "payment_ref": ""}
print(find_problems(messy, seen_ids={"TXN-1001"}))
print(find_problems(rows[0], seen_ids=set()))

['missing value: branch', "invalid number: quantity 'two' is not a whole number", 'invalid number: unit_price -520 is not positive', 'duplicate ID: TXN-1001 was already accepted earlier in this run']
[]


## Task 5: Clean records with loops, dicts, sets and comprehensions

`clean_row()` runs only on rows that passed `find_problems()`. It strips whitespace from every field
(dict comprehension), converts the columns in the `NUMERIC_COLUMNS` **set** to `int`, and adds the
computed `line_total = quantity x unit_price`. No pandas is used here.

In [8]:
def clean_row(row: dict) -> dict:
    """Return a normalised copy of a valid row (types converted, whitespace stripped)."""
    # dict comprehension: keep the output columns, strip spaces, turn None into ""
    cleaned = {col: (row.get(col) or "").strip() for col in CONSOLIDATED_COLUMNS if col != "line_total"}

    # convert the numeric columns (a set) to whole numbers
    for col in NUMERIC_COLUMNS:
        cleaned[col] = int(cleaned[col])

    # computed field
    cleaned["line_total"] = cleaned["quantity"] * cleaned["unit_price"]

    # return the columns in the agreed output order
    return {col: cleaned[col] for col in CONSOLIDATED_COLUMNS}


# testing: stray spaces are removed and numbers become ints
clean_row({"transaction_id": " TXN-2001 ", "date": "2026-10-04", "branch": "Nakuru Town ",
           "customer_id": "CUST-518", "product": " Kimbo 1kg", "quantity": " 3",
           "unit_price": "520 ", "payment_method": "M-Pesa", "payment_ref": "",
           "source_file": "transactions_2026-10-04.csv"})

{'transaction_id': 'TXN-2001',
 'date': '2026-10-04',
 'branch': 'Nakuru Town',
 'customer_id': 'CUST-518',
 'product': 'Kimbo 1kg',
 'quantity': 3,
 'unit_price': 520,
 'line_total': 1560,
 'payment_method': 'M-Pesa',
 'payment_ref': '',
 'source_file': 'transactions_2026-10-04.csv'}

## Tasks 6 and 7: Combine valid records, save rejected records

`run_pipeline()` processes the files in date order:

1. read the file and map any known column alias
2. if required columns are still missing, reject every row in the file with that reason
3. otherwise check each row: rows with problems go to the rejected list, clean rows are converted with
   `clean_row()` and their ID is added to `seen_ids`, so an exact repeat shows up only once
4. write both CSVs and check that **accepted + rejected = rows read**; the run stops with an error if
   any row went missing

The rejected file has `source_file`, `row_number`, `transaction_id` and all `reasons` joined with `; `,
followed by the original columns from every file.

In [9]:
def write_csv(rows: list[dict], path: Path, fieldnames: list[str]) -> None:
    """Write rows to CSV, creating parent folders as needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # "w" mode replaces the file on every run; extra keys (like row_number) are ignored
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def run_pipeline(input_dir: Path, output_dir: Path) -> dict:
    """Run every step and return the statistics used by build_summary()."""
    files = discover_files(input_dir)
    if not files:
        log.warning(f"No files matching {FILE_PATTERN} in {Path(input_dir).resolve()}")

    accepted, rejected = [], []
    seen_ids: set[str] = set()                                  # IDs accepted so far
    reason_counts = {label: 0 for label in CATEGORY_LABELS.values()}
    file_issues = []                                            # notes for the summary
    original_columns = []                                       # every input column, first-seen order
    rows_read = files_processed = 0

    def reject(row: dict, problems: list[str]) -> None:
        rejected.append({**row, "reasons": "; ".join(problems)})
        # set comprehension: count each category once per row
        for category in {problem.split(":")[0] for problem in problems}:
            reason_counts[CATEGORY_LABELS[category]] += 1

    for path in files:
        # 1. read, skipping (and reporting) a file that cannot be opened
        try:
            header, rows = read_transactions(path)
        except (OSError, UnicodeDecodeError, csv.Error) as exc:
            log.error(f"Could not read {path.name}: {exc}")
            file_issues.append(f"{path.name}: could not be read")
            continue
        rows_read += len(rows)

        # 2. map known aliases, then check the header
        header, renames = apply_aliases(header, rows)
        original_columns += [col for col in header if col not in original_columns]
        if renames:
            mapped = ", ".join(f"{new} (mapped from '{old}')" for old, new in renames.items())
            file_issues.append(f"{path.name}: missing {mapped}")
            log.warning(f"{path.name}: renamed columns {renames}")

        missing = validate_columns(header)
        if missing:
            missing_text = ", ".join(sorted(missing))
            file_issues.append(f"{path.name}: missing {missing_text} (whole file rejected)")
            log.warning(f"{path.name}: missing {missing_text}; all {len(rows)} rows rejected")
            for row in rows:
                reject(row, [f"file column issue: file is missing {missing_text}"])
            continue
        files_processed += 1

        # 3. check, then clean or reject, each row
        for row in rows:
            problems = find_problems(row, seen_ids)
            if problems:
                reject(row, problems)
            else:
                clean = clean_row(row)
                accepted.append(clean)
                seen_ids.add(clean["transaction_id"])

    # 4. nothing may disappear
    if len(accepted) + len(rejected) != rows_read:
        raise RuntimeError(f"Row count mismatch: {len(accepted)} accepted + {len(rejected)} "
                           f"rejected != {rows_read} read")

    output_dir = Path(output_dir)
    write_csv(accepted, output_dir / "consolidated_transactions.csv", CONSOLIDATED_COLUMNS)
    rejected_columns = REJECTED_LEAD_COLUMNS + [c for c in original_columns if c not in REJECTED_LEAD_COLUMNS]
    if any("extra_values" in row for row in rejected):
        rejected_columns.append("extra_values")
    write_csv(rejected, output_dir / "rejected_transactions.csv", rejected_columns)

    return {
        "files_discovered": len(files),
        "files_processed": files_processed,
        "file_issues": file_issues,
        "rows_read": rows_read,
        "rows_accepted": len(accepted),
        "rows_rejected": len(rejected),
        "reason_counts": reason_counts,
        "total_revenue": sum(row["line_total"] for row in accepted),
        "finished_at": datetime.now().isoformat(timespec="seconds"),
    }


# running the pipeline on the real data
stats = run_pipeline(INPUT_DIR, OUTPUT_DIR)
stats

{'files_discovered': 3,
 'files_processed': 3,
 'file_issues': ["transactions_2026-10-03.csv: missing quantity (mapped from 'qty')"],
 'rows_read': 43,
 'rows_accepted': 34,
 'rows_rejected': 9,
 'reason_counts': {'missing values': 4,
  'invalid numbers': 2,
  'duplicate IDs': 3,
  'malformed rows': 0,
  'file column issues': 0},
 'total_revenue': 22775,
 'finished_at': '2026-10-29T10:54:46'}

In [10]:
# previewing the rejected rows and their reasons
pd.read_csv(OUTPUT_DIR / "rejected_transactions.csv", dtype=str, keep_default_na=False)[
    ["source_file", "row_number", "transaction_id", "reasons"]
]

,source_file,row_number,transaction_id,reasons
0,transactions_2026-10-02.csv,4,TXN-1018,missing value: customer_id
1,transactions_2026-10-02.csv,5,TXN-1019,missing value: unit_price
2,transactions_2026-10-02.csv,6,TXN-1012,duplicate ID: TXN-1012 was already accepted ea...
3,transactions_2026-10-02.csv,8,TXN-1021,missing value: quantity
4,transactions_2026-10-02.csv,11,TXN-1023,duplicate ID: TXN-1023 was already accepted ea...
5,transactions_2026-10-02.csv,13,TXN-1025,missing value: branch
6,transactions_2026-10-02.csv,16,TXN-1028,invalid number: quantity 'two' is not a whole ...
7,transactions_2026-10-03.csv,7,TXN-1029,duplicate ID: TXN-1029 was already accepted ea...
8,transactions_2026-10-03.csv,8,TXN-1035,invalid number: quantity -4 is not positive


## Task 8: Generate a processing summary

`build_summary()` follows the layout in `expected_output.md`. A row with reasons in two categories is
counted once in each, so the category lines can add up to more than "Rows rejected"; with this
data they add up exactly. Extra categories (malformed rows, rejected files) are only listed when they
occur.

In [11]:
def build_summary(stats: dict) -> str:
    """Format the processing summary shown in the expected-output resource."""
    def line(label: str, value) -> str:
        return f"{label:<25}{value}"   # values start in column 26, as in the template

    issues = stats["file_issues"]
    issues_text = f"{len(issues)}" + (f"  ({'; '.join(issues)})" if issues else "")
    counts = stats["reason_counts"]

    lines = [
        "=== Processing summary ===",
        line("Files discovered:", stats["files_discovered"]),
        line("Files processed:", stats["files_processed"]),
        line("Files with column issues:", issues_text),
        line("Rows read:", stats["rows_read"]),
        line("Rows accepted:", stats["rows_accepted"]),
        line("Rows rejected:", stats["rows_rejected"]),
        line("  - missing values:", counts["missing values"]),
        line("  - invalid numbers:", counts["invalid numbers"]),
        line("  - duplicate IDs:", counts["duplicate IDs"]),
    ]
    # extra categories only when they happen
    for label in ["malformed rows", "file column issues"]:
        if counts[label]:
            lines.append(line(f"  - {label}:", counts[label]))
    lines += [
        line("Total revenue (KES):", f"{stats['total_revenue']:,}"),
        line("Run finished at:", stats["finished_at"]),
    ]
    return "\n".join(lines)


def save_summary(summary: str, output_dir: Path) -> Path:
    """Write the summary text to output/summary.txt and return the path."""
    path = Path(output_dir) / "summary.txt"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(summary + "\n", encoding="utf-8")
    return path


summary = build_summary(stats)
print(summary)
save_summary(summary, OUTPUT_DIR)

=== Processing summary ===
Files discovered:        3
Files processed:         3
Files with column issues:1  (transactions_2026-10-03.csv: missing quantity (mapped from 'qty'))
Rows read:               43
Rows accepted:           34
Rows rejected:           9
  - missing values:      4
  - invalid numbers:     2
  - duplicate IDs:       3
Total revenue (KES):     22,775
Run finished at:         2026-10-29T10:54:46


WindowsPath('output/summary.txt')

### Checking the numbers reconcile

The summary is checked against the two CSVs as they were written to disk.

In [12]:
# reading the outputs back
consolidated = pd.read_csv(OUTPUT_DIR / "consolidated_transactions.csv")
rejected = pd.read_csv(OUTPUT_DIR / "rejected_transactions.csv", dtype=str, keep_default_na=False)

checks = {
    "accepted rows match the consolidated CSV": len(consolidated) == stats["rows_accepted"],
    "rejected rows match the rejected CSV": len(rejected) == stats["rows_rejected"],
    "accepted + rejected = rows read": len(consolidated) + len(rejected) == stats["rows_read"],
    "no transaction_id appears twice": consolidated["transaction_id"].is_unique,
    "revenue matches the sum of line_total": consolidated["line_total"].sum() == stats["total_revenue"],
    "line_total = quantity x unit_price": (consolidated["line_total"]
                                           == consolidated["quantity"] * consolidated["unit_price"]).all(),
    "columns match expected_output.md": list(consolidated.columns) == CONSOLIDATED_COLUMNS,
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

PASS  accepted rows match the consolidated CSV
PASS  rejected rows match the rejected CSV
PASS  accepted + rejected = rows read
PASS  no transaction_id appears twice
PASS  revenue matches the sum of line_total
PASS  line_total = quantity x unit_price
PASS  columns match expected_output.md


## Error handling

Two cases the real data does not contain, run against throwaway copies so `data/` and `output/` are
untouched:

1. a file with a required column that has no known alias: the whole file is rejected, row by row,
   and the counts still reconcile
2. a row with more values than the header (rejected as malformed), and a file that is not valid
   text (reported and skipped, while the other file is still processed)
3. an input folder that does not exist: one clear error message

In [13]:
# 1. a fourth day's file with no payment_method column at all
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    for path in files:
        shutil.copy(path, tmp)
    (tmp / "transactions_2026-10-04.csv").write_text(
        "transaction_id,date,branch,customer_id,product,quantity,unit_price\n"
        "TXN-1041,2026-10-04,Nakuru Town,CUST-518,Kimbo 1kg,1,520\n"
        "TXN-1042,2026-10-04,Kisumu Mega,CUST-087,Unga Pembe 2kg,2,210\n"
    )
    test_stats = run_pipeline(tmp, tmp / "output")
    print()
    print(build_summary(test_stats))


=== Processing summary ===
Files discovered:        4
Files processed:         3
Files with column issues:2  (transactions_2026-10-03.csv: missing quantity (mapped from 'qty'); transactions_2026-10-04.csv: missing payment_method (whole file rejected))
Rows read:               45
Rows accepted:           34
Rows rejected:           11
  - missing values:      4
  - invalid numbers:     2
  - duplicate IDs:       3
  - file column issues:  2
Total revenue (KES):     22,775
Run finished at:         2026-10-29T10:54:46


In [14]:
# 3. a row with more values than the header, and a file that is not valid UTF-8 text
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    (tmp / "transactions_2026-10-05.csv").write_text(
        "transaction_id,date,branch,customer_id,product,quantity,unit_price,payment_method,payment_ref\n"
        "TXN-1050,2026-10-05,Nakuru Town,CUST-518,Kimbo 1kg,1,520,Cash,,EXTRA\n"
        "TXN-1051,2026-10-05,Kisumu Mega,CUST-087,Unga Pembe 2kg,2,210,Cash,\n"
    )
    (tmp / "transactions_2026-10-06.csv").write_bytes(b"transaction_id,date\n\xff\xfe\x00broken")
    test_stats = run_pipeline(tmp, tmp / "output")
    print(build_summary(test_stats))
    print()
    print(pd.read_csv(tmp / "output" / "rejected_transactions.csv", dtype=str, keep_default_na=False)[
        ["transaction_id", "reasons", "extra_values"]])

ERROR: Could not read transactions_2026-10-06.csv: 'utf-8' codec can't decode byte 0xff in position 20: invalid start byte


=== Processing summary ===
Files discovered:        2
Files processed:         1
Files with column issues:1  (transactions_2026-10-06.csv: could not be read)
Rows read:               2
Rows accepted:           1
Rows rejected:           1
  - missing values:      0
  - invalid numbers:     0
  - duplicate IDs:       0
  - malformed rows:      1
Total revenue (KES):     420
Run finished at:         2026-10-29T10:54:46

  transaction_id                                            reasons  \
0       TXN-1050  malformed row: more values than columns in the...   

  extra_values  
0        EXTRA  


In [15]:
# 4. an input folder that does not exist
try:
    run_pipeline(Path("no_such_folder"), OUTPUT_DIR)
except FileNotFoundError as exc:
    print(f"ERROR: {exc}")

ERROR: Input folder not found: C:\Users\Stanley\Desktop\Lux_Dev\Multi-File Data Processing Pipeline\no_such_folder


## Full pipeline

`main()` runs everything end to end: process the files, print the summary and save it.

In [16]:
def main(input_dir: Path = INPUT_DIR, output_dir: Path = OUTPUT_DIR) -> None:
    try:
        stats = run_pipeline(input_dir, output_dir)
    except FileNotFoundError as exc:
        log.error(exc)
        return
    summary = build_summary(stats)
    print(summary)
    save_summary(summary, output_dir)


# running the full pipeline
main()

=== Processing summary ===
Files discovered:        3
Files processed:         3
Files with column issues:1  (transactions_2026-10-03.csv: missing quantity (mapped from 'qty'))
Rows read:               43
Rows accepted:           34
Rows rejected:           9
  - missing values:      4
  - invalid numbers:     2
  - duplicate IDs:       3
Total revenue (KES):     22,775
Run finished at:         2026-10-29T10:54:46
